<center><a href="https://www.nvidia.com/dli"> <img src="https://drive.google.com/uc?id=1J3bpGyJcz-7uOFkUNhvOBiReBCk-sUwR" width="200" </a></center>

# Assessment

In this assessment, you will train a new model that is able to recognize fresh and rotten fruit. You will need to get the model to a validation accuracy of `92%` in order to pass the assessment, though we challenge you to do even better if you can. You will have the use the skills that you learned in the previous exercises. Specifically, we suggest using some combination of transfer learning, data augmentation, and fine tuning. Once you have trained the model to be at least 92% accurate on the validation dataset, save your model, and then assess its accuracy. Let's get started!

In [1]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [2]:
%cd /content/gdrive/MyDrive/'Colab Notebooks'/Fundamentals_Deep_Learning

!ls

/content/gdrive/MyDrive/Colab Notebooks/Fundamentals_Deep_Learning
001_tensorflow.ipynb  04a_asl_augmentation.ipynb	 asl_model.h5
00_jupyterlab.ipynb   04b_asl_predictions.ipynb		 asl_model.keras
01_mnist.ipynb	      05a_doggy_door.ipynb		 data
02_asl.ipynb	      05b_presidential_doggy_door.ipynb  desktop.ini
03_asl_cnn.ipynb      06_headline_generator.ipynb	 images


In [3]:
path = "/content/gdrive/MyDrive/Colab Notebooks/Fundamentals_Deep_Learning/data/fruits/"


## The Dataset

In this exercise, you will train a model to recognize fresh and rotten fruits. The dataset comes from [Kaggle](https://www.kaggle.com/sriramr/fruits-fresh-and-rotten-for-classification), a great place to go if you're interested in starting a project after this class. The dataset structure is in the `data/fruits` folder. There are 6 categories of fruits: fresh apples, fresh oranges, fresh bananas, rotten apples, rotten oranges, and rotten bananas. This will mean that your model will require an output layer of 6 neurons to do the categorization successfully. You'll also need to compile the model with `categorical_crossentropy`, as we have more than two categories.

<img src="https://drive.google.com/uc?id=1ryuTigL-jyigpICE-veqLZvPz1DCg4ml" style="width: 600px;">

## Load ImageNet Base Model

We encourage you to start with a model pretrained on ImageNet. Load the model with the correct weights, set an input shape, and choose to remove the last layers of the model. Remember that images have three dimensions: a height, and width, and a number of channels. Because these pictures are in color, there will be three channels for red, green, and blue. We've filled in the input shape for you. This cannot be changed or the assessment will fail. If you need a reference for setting up the pretrained model, please take a look at [notebook 05b] where we implemented transfer learning.

In [4]:
from tensorflow import keras

base_model = keras.applications.VGG16(
    weights='imagenet',
    input_shape=(224, 224, 3),
    include_top=False)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Freeze Base Model

Next, we suggest freezing the base model, as done in [notebook 05b]. This is done so that all the learning from the ImageNet dataset does not get destroyed in the initial training.

In [5]:
# Freeze base model
base_model.trainable = False

## Add Layers to Model

Now it's time to add layers to the pretrained model. [Notebook 05b] can be used as a guide. Pay close attention to the last dense layer and make sure it has the correct number of neurons to classify the different types of fruit.

In [6]:
# Create inputs with correct shape
inputs = keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)

# Add pooling layer or flatten layer
x = keras.layers.GlobalAveragePooling2D()(x) # reduz a dimensionalidade.
x = keras.layers.Dropout(0.2)(x)  # Regularização
# Add final dense layer
outputs = keras.layers.Dense(6, activation = 'softmax')(x)

# Combine inputs and outputs to create model
model = keras.Model(inputs=inputs, outputs=outputs)

In [7]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │         3,078 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,717,766 (56.14 MB)

 Trainable params: 3,078 (12.02 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

## Compile Model

Now it's time to compile the model with loss and metrics options. Remember that we're training on a number of different categories, rather than a binary classification problem.

In [8]:
model.compile(loss = 'categorical_crossentropy' , metrics = ['accuracy'])

## Augment the Data

If you'd like, try to augment the data to improve the dataset. Feel free to look at [notebook 04a] and [notebook 05b] for augmentation examples. There is also documentation for the [Keras ImageDataGenerator class](https://keras.io/api/preprocessing/image/#imagedatagenerator-class). This step is optional, but it may be helpful to get to 92% accuracy.

In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen_train = ImageDataGenerator(
    rescale=1.0/255, # Normalização
    rotation_range=20,  # randomly rotate images in the range (degrees, 0 to 180)
    zoom_range=0.2,  # Randomly zoom image
    width_shift_range=0.15,  # randomly shift images horizontally (fraction of total width)
    height_shift_range=0.12,  # randomly shift images vertically (fraction of total height)
    horizontal_flip=False,  # randomly flip images horizontally
    vertical_flip=False, # Don't randomly flip images vertically
)
datagen_valid = ImageDataGenerator(rescale=1./255) # Normalizando

## Load Dataset

Now it's time to load the train and validation datasets. Pick the right folders, as well as the right `target_size` of the images (it needs to match the height and width input of the model you've created). For a reference, check out [notebook 05b].

In [10]:
# load and iterate training dataset
train_it = datagen_train.flow_from_directory(
    path+'train', # treino
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=32)

# load and iterate validation dataset
valid_it = datagen_valid.flow_from_directory(
    path+'valid', # validaçao
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=32)

Found 1182 images belonging to 6 classes.
Found 329 images belonging to 6 classes.


## Train the Model

Time to train the model! Pass the `train` and `valid` iterators into the `fit` function, as well as setting the desired number of epochs.

In [11]:
model.fit(
    train_it,
    validation_data=valid_it,
    steps_per_epoch=train_it.samples // train_it.batch_size,
    validation_steps=valid_it.samples // valid_it.batch_size,
    epochs=20
)

Epoch 1/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 286s 8s/step - accuracy: 0.3000 - loss: 1.6860 - val_accuracy: 0.4781 - val_loss: 1.5255
Epoch 2/20
 1/36 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4375 - loss: 1.5774

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 133ms/step - accuracy: 0.4375 - loss: 1.5774 - val_accuracy: 0.5219 - val_loss: 1.5214
Epoch 3/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 21s 580ms/step - accuracy: 0.4861 - loss: 1.4970 - val_accuracy: 0.5250 - val_loss: 1.3551
Epoch 4/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.3750 - loss: 1.5184 - val_accuracy: 0.5312 - val_loss: 1.3527
Epoch 5/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 21s 576ms/step - accuracy: 0.5713 - loss: 1.3474 - val_accuracy: 0.6656 - val_loss: 1.2250
Epoch 6/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.4688 - loss: 1.3365 - val_accuracy: 0.6875 - val_loss: 1.2188
Epoch 7/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 21s 578ms/step - accuracy: 0.6365 - loss: 1.2439 - val_accuracy: 0.6844 - val_loss: 1.1074
Epoch 8/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - accuracy: 0.5625 - loss: 1.1718 - val_accuracy: 0.7063 - val_loss: 1.1032
Epoch 9/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 21s 577ms/step - accuracy: 0.6887 - loss: 1.1399 - val_accuracy: 0.7219 - val

## Unfreeze Model for Fine Tuning

If you have reached 92% validation accuracy already, this next step is optional. If not, we suggest fine tuning the model with a very low learning rate.

In [12]:
# Unfreeze the base model
base_model.trainable = True

# Compile the model with a low learning rate
model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-5), # Learning rate baixo
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [13]:
model.fit(train_it,
          validation_data=valid_it,
          steps_per_epoch=train_it.samples // train_it.batch_size,
          validation_steps=valid_it.samples // valid_it.batch_size,
          epochs=10) # Mais 10 épocas com learning rate baixo

Epoch 1/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 54s 1s/step - accuracy: 0.8409 - loss: 0.4676 - val_accuracy: 0.8938 - val_loss: 0.2967
Epoch 2/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 0.8438 - loss: 0.6029 - val_accuracy: 0.8906 - val_loss: 0.3075
Epoch 3/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 22s 598ms/step - accuracy: 0.8948 - loss: 0.2745 - val_accuracy: 0.9406 - val_loss: 0.1670
Epoch 4/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.8125 - loss: 0.3753 - val_accuracy: 0.9250 - val_loss: 0.1983
Epoch 5/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 21s 595ms/step - accuracy: 0.9278 - loss: 0.2079 - val_accuracy: 0.9656 - val_loss: 0.1184
Epoch 6/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - accuracy: 1.0000 - loss: 0.1187 - val_accuracy: 0.9688 - val_loss: 0.1229
Epoch 7/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 22s 598ms/step - accuracy: 0.9409 - loss: 0.1619 - val_accuracy: 0.9375 - val_loss: 0.1979
Epoch 8/10
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - accuracy: 0.8750 - loss: 0.3281 - val_accuracy: 0.956

## Evaluate the Model

Hopefully, you now have a model that has a validation accuracy of 92% or higher. If not, you may want to go back and either run more epochs of training, or adjust your data augmentation.

Once you are satisfied with the validation accuracy, evaluate the model by executing the following cell. The evaluate function will return a tuple, where the first value is your loss, and the second value is your accuracy. To pass, the model will need have an accuracy value of `92% or higher`.

In [14]:
model.evaluate(valid_it, steps=valid_it.samples//valid_it.batch_size)

10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 218ms/step - accuracy: 0.9594 - loss: 0.1012


[0.10121749341487885, 0.9593750238418579]